# FineTunePilot — Stage 4: one real agent session on Kaggle

The agent evaluates the base model, asks Claude for a first QLoRA config, checks it against the
VRAM estimate, trains (with OOM recovery), evaluates, asks Claude for ONE change, and repeats
while the score improves — up to 3 runs / 90 minutes. Then it writes a report.

**Costs:** this calls the Anthropic API (billed to your key) for 1 plan + up to 3 diagnoses.
**Before you start:** Accelerator → **GPU T4**, **Internet on**, and add your key as a Kaggle
Secret named `ANTHROPIC_API_KEY` (Add-ons → Secrets). The key is never printed or saved.

## 1. GPU, code and install

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /kaggle/working
if not os.path.isdir("Finetune-pilot-ai"):
    !git clone --depth 1 --branch claude/zen-meitner-hetiul https://github.com/manojastro/Finetune-pilot-ai.git
%cd Finetune-pilot-ai
!pip install -q -e ".[gpu]"

## 2. One GPU, and the API key from Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("key loaded:", bool(os.environ["ANTHROPIC_API_KEY"]))

## 3. Dry run first (no API calls, ~10 s)
Checks the install with a rule-based fake LLM and simulated training.

In [ ]:
!finetunepilot agent --objective "tool calling" --dry-run --yes --max-runs 2

## 4. The real session
`--yes` approves each run automatically (notebooks can't answer the y/n prompt). The LLM model is set in `configs/agent.yaml`.

In [ ]:
!finetunepilot agent --objective "Pick the right tool and produce exactly the right JSON arguments" --model Qwen/Qwen2.5-0.5B-Instruct --dataset data/sample --gpu t4 --max-runs 3 --max-minutes 90 --yes

## 5. Show the report
Copy this back into Claude Code.

In [ ]:
import glob
report = sorted(glob.glob("reports/s-*.md"), key=os.path.getmtime)[-1]
print(open(report).read())

If the session stopped early (crash, timeout), continue it with `finetunepilot agent --resume <session_id> --yes`.